<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/10_vector_rag_faiss/vector_rag_faiss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U sentence-transformers transformers faiss-cpu sentencepiece --quiet

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
documents = [
    "Paris is the capital city of France.",
    "The Eiffel Tower is located in Paris.",
    "Mars is known as the Red Planet.",
    "William Shakespeare wrote Romeo and Juliet."
]

In [ ]:
doc_embeddings = embed_model.encode(documents)

In [ ]:
import faiss
import numpy as np

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

In [ ]:
question = "Which city is the capital of France?"

question_embedding = embed_model.encode([question])

In [10]:
D, I = index.search(np.array(question_embedding), k=1)

context = documents[I[0][0]]

print("RETRIEVED CONTEXT:")
print(context)

RETRIEVED CONTEXT:
Paris is the capital city of France.


In [11]:
prompt = f"""
Use the following context to answer the question.

Context: {context}

Question: {question}
"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=20)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\nFINAL ANSWER:")
print(answer)


FINAL ANSWER:
Paris
